In [ ]:
# ============================================================
# أول شي نثبّت الـ
# [libraries]
# الأساسية اللي يحتاجها المشروع كله
# [tensorflow]:
# حق بناء وتدريب نماذج الـ
# [deep learning]
# [pandas]:
# حق قراءة ملفات الـ
# [CSV]
# والجداول ومعالجتها
# [sklearn]:
# حق التطبيع وتجهيز البيانات قبل التدريب
# [numpy]:
# حق الحسابات الرياضية والمصفوفات
# [matplotlib]
# [seaborn]
# حق رسم المخططات وتصوير النتائج
# في الآخر نتحقق من إصدار الـ
# [TensorFlow]
# عشان نضمن التوافق
# ============================================================
%pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json as json_lib
import pickle
import os
import zipfile
import warnings
from datetime import datetime

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

warnings.filterwarnings('ignore')
print(f'TensorFlow version: {tf.__version__}')
print('Setup complete!')

In [ ]:
# ============================================================
# إعدادات ملفات الدفتر وملفات CSV
# ============================================================

# المطلوب من المستخدم رفع ملفات CSV الخاصة بالنشاط
# ملاحظة: الكود يحتاج ملفات أساسية عشان يشتغل بدون مشاكل
# ------------------------------------------------------------

# ملفات logon.csv و device.csv تعتبر أساسية ومهمة جداً
# ملفات file.csv و http.csv و ldap.csv اختيارية وتضيف خصائص إضافية

# لو الملفات ثقيلة، استخدم النسخ المصغّرة التي تنتهي بـ *_small.csv
# ملاحظة: الكود بيطبع أسماء الملفات بعد الرفع عشان نتأكد
# ============================================================
from google.colab import files

print('Upload your CSV files (logon, device, file, http, ldap)')
print('You can upload the *_small.csv versions too.\n')

uploaded = files.upload()
print(f'\nUploaded {len(uploaded)} files: {list(uploaded.keys())}')

In [ ]:
# ============================================================
# قراءة ملفات النشاط وتحضير أعمدة المستخدم والتاريخ
# [find_csv]
# [logon]
# [device]
# [file]
# [http]
# [ldap]
# نقرأ كل ملف اترفع ونحوّل عمود التاريخ لنوع datetime
# عندنا دالة find_csv تدور على الملف بالاسم بغض النظر عن الحروف الكبيرة
# لو logon ما اترفع، نوقف الكل — هذا الملف أساسي وما نكمل بدونه
# لو device أو file أو http ما اترفعوا، نكمل ونحط أصفار بدل بياناتهم
# ملف ldap اختياري بس يضيف معلومات الوظيفة والقسم حق النموذج
# ملاحظة: بعض ملفات http.csv ما فيها عمود user — في هالحالة نتجاهله تلقائياً
# ============================================================
def find_csv(name_patterns, uploaded_files):
    """تبحث عن ملف مرفوع يطابق أي كلمة من الكلمات المطلوبة في الاسم."""
    for key in uploaded_files.keys():
        key_lower = key.lower()
        for pattern in name_patterns:
            if pattern.lower() in key_lower:
                return key
    return None

def get_user_col(df):
    """ترجع اسم عمود المستخدم إذا كان موجوداً بصيغة user أو user_id."""
    if 'user' in df.columns:
        return 'user'
    if 'user_id' in df.columns:
        return 'user_id'
    return None

def normalize_user_col(df):
    """توحّد اسم عمود المستخدم إلى user حتى تستخدمه كل خطوات المعالجة بنفس الطريقة."""
    col = get_user_col(df)
    if col is None:
        return None
    if col == 'user_id':
        df = df.rename(columns={'user_id': 'user'})
    return df

logon_file = find_csv(['logon'], uploaded)
if logon_file:
    logon_df = pd.read_csv(logon_file)
    logon_df = normalize_user_col(logon_df)
    if logon_df is None:
        raise ValueError('logon.csv has no user/user_id column!')
    logon_df['date'] = pd.to_datetime(logon_df['date'])
    print(f'Logon: {len(logon_df):,} rows | Users: {logon_df["user"].nunique()} | {logon_df["date"].min().date()} to {logon_df["date"].max().date()}')
else:
    raise FileNotFoundError('logon.csv is required! Please re-upload.')

device_file = find_csv(['device'], uploaded)
if device_file:
    device_df = pd.read_csv(device_file)
    device_df = normalize_user_col(device_df)
    if device_df is None:
        print('WARNING: device.csv has no user column — skipping.')
    else:
        device_df['date'] = pd.to_datetime(device_df['date'])
        print(f'Device: {len(device_df):,} rows | Users: {device_df["user"].nunique()}')
else:
    print('WARNING: device.csv not found. USB features will be zero.')
    device_df = None

file_file = find_csv(['file'], uploaded)
if file_file:
    file_df = pd.read_csv(file_file)
    file_df = normalize_user_col(file_df)
    if file_df is None:
        print('WARNING: file.csv has no user column — skipping.')
    else:
        file_df['date'] = pd.to_datetime(file_df['date'])
        print(f'File: {len(file_df):,} rows | Users: {file_df["user"].nunique()}')
else:
    print('INFO: file.csv not found. File features will be zero.')
    file_df = None

http_file = find_csv(['http'], uploaded)
if http_file:
    http_df = pd.read_csv(http_file)
    http_df = normalize_user_col(http_df)
    if http_df is None:
        print('INFO: http.csv has no user column — HTTP features will be skipped.')
    else:
        http_df['date'] = pd.to_datetime(http_df['date'])
        print(f'HTTP: {len(http_df):,} rows | Users: {http_df["user"].nunique()}')
else:
    print('INFO: http.csv not found. HTTP features will be zero.')
    http_df = None

ldap_file = find_csv(['ldap'], uploaded)
if ldap_file:
    ldap_df = pd.read_csv(ldap_file)
    print(f'LDAP: {len(ldap_df):,} rows | Roles: {ldap_df["role"].nunique() if "role" in ldap_df.columns else "N/A"}')
else:
    print('INFO: ldap.csv not found. Context features will be limited.')
    ldap_df = None

print('\nData loading complete!')

## 2. Data Exploration

In [ ]:
# ============================================================
# نطّلع على نماذج من البيانات اللي اترفعت عشان نتأكد منها قبل ما نكمل
# [Logon]
# [Logoff]
# [Connect]
# نشوف وش الأنشطة الموجودة زي فوق
# نتأكد كذلك إن الأعمدة زينة وإن التواريخ اتحوّلت صح
# هذي الخطوة تساعدنا نكتشف أي مشكلة في البيانات من البداية
# ============================================================
print('=' * 60)
print('LOGON DATA')
print('=' * 60)
print(f'Activities: {logon_df["activity"].value_counts().to_dict()}')
print(logon_df.head(3))
print()

if device_df is not None:
    print('=' * 60)
    print('DEVICE DATA')
    print('=' * 60)
    print(f'Activities: {device_df["activity"].value_counts().to_dict()}')
    print(device_df.head(3))
    print()

if file_df is not None:
    print('=' * 60)
    print('FILE DATA')
    print('=' * 60)
    print(f'Columns: {list(file_df.columns)}')
    print(file_df.head(3))
    print()

if http_df is not None:
    print('=' * 60)
    print('HTTP DATA')
    print('=' * 60)
    print(http_df.head(3))


## 3. Feature Engineering

Create **daily behavioral features** for each user from all log sources.

Features extracted:
- **Logon:** count, unique PCs, after-hours logons, weekend logons, first/last hour, session span
- **Device:** USB connects, after-hours USB usage
- **File:** file actions, unique files, copies to removable media
- **HTTP:** request count, unique domains


In [ ]:
# ============================================================
# نعرّف 4 دوال، كل وحدة تسحب الـ [features] من مصدر بيانات مختلف
# [extract_logon_features]:
# عدد الدخولات، كم جهاز استخدم، دخولات بعد الدوام،
#   دخولات في العطلة، أول وآخر دخول في اليوم، ومدة الجلسة بالساعات
# [extract_device_features]:
# كم مرة وصّل [USB] وكم منها كانت بعد الدوام
# [extract_file_features]:
# عدد الملفات اللي تعامل معها ومنها كم نسخ على [USB]
# [extract_http_features]:
# كم طلب [HTTP] سوّى وكم نطاق مختلف دخل عليه
# كل دالة ترجع جدول يومي — صف لكل (مستخدم + يوم) فيه الأرقام مجمّعة
# ============================================================
def extract_logon_features(df):
    """تستخرج مؤشرات الدخول اليومية لكل مستخدم مثل العدد والأجهزة والوقت بعد الدوام."""
    df = df.copy()
    df['day'] = df['date'].dt.date
    df['hour'] = df['date'].dt.hour
    df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
    df['is_after_hours'] = ((df['hour'] < 7) | (df['hour'] >= 19)).astype(int)

    logons = df[df['activity'] == 'Logon']

    features = logons.groupby(['user', 'day']).agg(
        logon_count=('id', 'count'),
        unique_pc_count=('pc', 'nunique'),
        after_hours_logon=('is_after_hours', 'sum'),
        weekend_logon=('is_weekend', 'max'),
        first_logon_hour=('hour', 'min'),
        last_logon_hour=('hour', 'max'),
    ).reset_index()

    features['logon_span_hours'] = (features['last_logon_hour'] - features['first_logon_hour']).clip(lower=0)
    return features


def extract_device_features(df):
    """تستخرج مؤشرات استخدام الأجهزة الخارجية وخصوصاً توصيل USB بعد الدوام."""
    if df is None or len(df) == 0:
        return None
    df = df.copy()
    df['day'] = df['date'].dt.date
    df['hour'] = df['date'].dt.hour
    df['is_after_hours'] = ((df['hour'] < 7) | (df['hour'] >= 19)).astype(int)

    connects = df[df['activity'] == 'Connect']
    features = connects.groupby(['user', 'day']).agg(
        usb_connect_count=('id', 'count'),
        usb_after_hours=('is_after_hours', 'sum'),
    ).reset_index()
    return features


def extract_file_features(df):
    """تستخرج مؤشرات نشاط الملفات وعدد الملفات الفريدة والنسخ إلى وسائط قابلة للإزالة."""
    if df is None or len(df) == 0:
        return None
    df = df.copy()
    df['day'] = df['date'].dt.date

    features = df.groupby(['user', 'day']).agg(
        file_action_count=('id', 'count'),
        unique_file_count=('filename', 'nunique'),
    ).reset_index()

    # يتحقق من وجود عمود النسخ إلى وسائط قابلة للإزالة في بيانات CERT
    if 'to_removable_media' in df.columns:
        df['_to_rm'] = df['to_removable_media'].apply(
            lambda x: str(x).strip().lower() == 'true'
        )
        to_rm = df.groupby(['user', 'day']).agg(
            file_to_removable=('_to_rm', 'sum')
        ).reset_index()
        features = features.merge(to_rm, on=['user', 'day'], how='left')
    else:
        # إذا لم يوجد العمود، نعتبر كل سجل ملف كاحتمال نسخ حسب وصف بيانات CERT
        features['file_to_removable'] = features['file_action_count']

    return features


def extract_http_features(df):
    """تستخرج مؤشرات تصفح الويب مثل عدد الطلبات وعدد النطاقات المختلفة."""
    if df is None or len(df) == 0:
        return None
    df = df.copy()
    df['day'] = df['date'].dt.date
    df['domain'] = df['url'].str.extract(r'(?:https?://)?([^/]+)', expand=False)

    features = df.groupby(['user', 'day']).agg(
        http_count=('id', 'count'),
        unique_domains=('domain', 'nunique'),
    ).reset_index()
    return features


print('Feature extraction functions defined.')

In [ ]:
# ============================================================
# الحين نشغّل الدوال الأربعة على كل مصدر بيانات عندنا
# بعدين ندمّج كل النتائج في جدول واحد على أساس (مستخدم + يوم) —
# [merge]
# كل موظف يطلع له صف يومي فيه جميع الـ 
# [features] 
# مجمّعة مع بعض
# لو مصدر بيانات ما اترفع، أعمدته تتملى بأصفار تلقائياً —
# [fillna(0)]
# في الآخر يطبع عدد الصفوف والأعمدة وأسماء الـ 
# [features] 
# الكاملة
# ============================================================
print('Extracting logon features...')
logon_features = extract_logon_features(logon_df)
print(f'  Shape: {logon_features.shape}')

print('Extracting device features...')
device_features = extract_device_features(device_df)
if device_features is not None:
    print(f'  Shape: {device_features.shape}')

print('Extracting file features...')
file_features = extract_file_features(file_df)
if file_features is not None:
    print(f'  Shape: {file_features.shape}')

print('Extracting HTTP features...')
http_features = extract_http_features(http_df)
if http_features is not None:
    print(f'  Shape: {http_features.shape}')

print('\nMerging all features...')
daily_features = logon_features.copy()

if device_features is not None:
    daily_features = daily_features.merge(device_features, on=['user', 'day'], how='left')

if file_features is not None:
    daily_features = daily_features.merge(file_features, on=['user', 'day'], how='left')

if http_features is not None:
    daily_features = daily_features.merge(http_features, on=['user', 'day'], how='left')

daily_features = daily_features.fillna(0)

feature_cols = [c for c in daily_features.columns if c not in ['user', 'day']]
print(f'\nDaily features: {daily_features.shape[0]:,} rows x {len(feature_cols)} features')
print(f'Features: {feature_cols}')
print(f'Unique users: {daily_features["user"].nunique()}')
daily_features.head(10)

In [ ]:
# ============================================================
# نضيف بيانات الـ [LDAP] اللي فيها الوظيفة والقسم حق كل موظف
# [Label Encoding]
# نحوّل الأدوار والأقسام لأرقام عشان النموذج يفهمها
# نحدد كمان مين
# [Admin]
# لأن صلاحياته أكبر وسلوكه يختلف عن الموظف العادي
# هذي الـ
# [context features]
# تساعد النموذج يفهم السياق الوظيفي لكل شخص
# نحفظ الـ
# [encoders]
# عشان نستخدمهم بنفس الطريقة وقت النشر على بيانات جديدة
# ============================================================
context_data = {}

if ldap_df is not None:
    user_col = 'user_id' if 'user_id' in ldap_df.columns else 'user'
    ldap_unique = ldap_df.drop_duplicates(subset=[user_col], keep='last')

    # ترميز المسمى الوظيفي وتحويله إلى رقم قابل للاستخدام داخل النموذج
    if 'role' in ldap_unique.columns:
        role_enc = LabelEncoder()
        ldap_unique = ldap_unique.copy()
        ldap_unique['role_encoded'] = role_enc.fit_transform(ldap_unique['role'].fillna('Unknown'))
        ldap_unique['is_admin'] = ldap_unique['role'].str.contains('Admin', case=False, na=False).astype(int)

        role_map = ldap_unique.set_index(user_col)['role_encoded'].to_dict()
        admin_map = ldap_unique.set_index(user_col)['is_admin'].to_dict()

        daily_features['role_encoded'] = daily_features['user'].map(role_map).fillna(0)
        daily_features['is_admin'] = daily_features['user'].map(admin_map).fillna(0)

        context_data['role_encoder'] = role_enc
        print(f'Added role features. Roles found: {list(role_enc.classes_)}')

    # ترميز القسم وتحويله إلى رقم حتى يدخل ضمن خصائص التدريب
    if 'department' in ldap_unique.columns:
        dept_enc = LabelEncoder()
        ldap_unique['dept_encoded'] = dept_enc.fit_transform(ldap_unique['department'].fillna('Unknown'))
        dept_map = ldap_unique.set_index(user_col)['dept_encoded'].to_dict()
        daily_features['dept_encoded'] = daily_features['user'].map(dept_map).fillna(0)

        context_data['dept_encoder'] = dept_enc
        print(f'Added department features. Departments: {ldap_unique["department"].nunique()}')

    # حفظ معلومات المستخدم حتى نستطيع عرض الدور والقسم لاحقاً في لوحة التحكم
    user_info = ldap_unique[[user_col, 'employee_name', 'role']].copy()
    if 'department' in ldap_unique.columns:
        user_info['department'] = ldap_unique['department']
    context_data['user_info'] = user_info
    print(f'User info saved for {len(user_info)} users')
else:
    print('No LDAP data — skipping context features.')

feature_cols = [c for c in daily_features.columns if c not in ['user', 'day']]
print(f'\nFinal feature set ({len(feature_cols)} features): {feature_cols}')

## 4. Data Preparation

Scale features and prepare data for both models.


In [ ]:
# ============================================================
# نطبّق الـ
# [StandardScaler]
# على جميع الـ
# [features]
# عشان نسوّيها
# ضروري عشان الأرقام الكبيرة زي عدد الدخولات (100) ما تطغى
# على الأرقام الصغيرة زي نسبة السهر (0.3) — كلها تصير على نفس السكيل
# الـ
# [Scaler]
# يحسب المتوسط والانحراف المعياري ويحوّل كل شي لـ
# [z-score]
# لازم نحفظه عشان نطبّقه بنفس الطريقة على أي بيانات جديدة وقت النشر
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(daily_features[feature_cols].values)

print(f'Scaled data shape: {X_scaled.shape}')
print(f'Means (should be ~0): {np.round(X_scaled.mean(axis=0), 2)}')
print(f'Stds  (should be ~1): {np.round(X_scaled.std(axis=0), 2)}')

daily_features['day_idx'] = pd.factorize(daily_features['day'])[0]

## 5. Autoencoder Model

The Autoencoder learns to **reconstruct normal behavior**. If a day's behavior has a high reconstruction error, it is flagged as anomalous.

Architecture: `Input → Dense → Dropout → Bottleneck → Dense → Dropout → Output`


In [ ]:
# ============================================================
# نبني نموذج الـ
# [Autoencoder]
# — يتعلم يعيد بناء سلوك يوم عادي
# المعمارية: [Input] ← [Dense] كبير ← [Dropout] ← [Bottleneck] ← [Dense] ← [Output]
# الـ
# [Encoder]
# يضغط البيانات لتمثيل مضغوط ([compressed representation])
# الـ
# [Decoder]
# يحاول يرجع البيانات الأصلية من التمثيل المضغوط
# لو اليوم شاذ، النموذج ما يقدر يعيد بناؤه زين — وهذا هو اللي نبيه يكشفه
# الـ
# [Dropout]
# بنسبة 20% يمنع الـ
# [overfitting]
# أثناء التدريب
# ============================================================
# بناء نموذج Autoencoder
input_dim = X_scaled.shape[1]
encoding_dim = max(4, input_dim // 3)

ae_input = keras.Input(shape=(input_dim,), name='ae_input')

# جزء الضغط الذي يتعلم تمثيلاً مختصراً للسلوك اليومي
x = layers.Dense(input_dim * 2, activation='relu', name='enc_1')(ae_input)
x = layers.Dropout(0.2)(x)
x = layers.Dense(encoding_dim, activation='relu', name='bottleneck')(x)

# جزء إعادة البناء الذي يحاول استرجاع الخصائص الأصلية
x = layers.Dense(input_dim * 2, activation='relu', name='dec_1')(x)
x = layers.Dropout(0.2)(x)
ae_output = layers.Dense(input_dim, activation='linear', name='ae_output')(x)

autoencoder = Model(ae_input, ae_output, name='Autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')

print(f'Input dim: {input_dim} | Bottleneck: {encoding_dim}')
autoencoder.summary()

In [ ]:
# ============================================================
# ندرّب الـ
# [Autoencoder]
# على كل البيانات — 50 جولة تدريب ([epoch])
# التدريب بدون إشراف
# [unsupervised]
# — ما نحتاج نقول مين مشبوه مسبقاً
# النموذج يتعلم لوحده يعيد بناء السلوك العادي اللي يشوفه في معظم البيانات
# نحجز 10% للتحقق
# [validation]
# عشان نشوف الأداء على بيانات ما شافها
# بعد التدريب نرسم منحنى الـ
# [Loss]
# — لو [val_loss] قريب من [train_loss] يعني التدريب زين
# ============================================================
# تدريب نموذج Autoencoder على إعادة بناء نفس المدخلات
history_ae = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=1,
)

# رسم خسارة التدريب والتحقق لمراقبة جودة التعلم
plt.figure(figsize=(10, 4))
plt.plot(history_ae.history['loss'], label='Train Loss')
plt.plot(history_ae.history['val_loss'], label='Val Loss')
plt.title('Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# نحسب خطأ إعادة البناء
# ([reconstruction error]) لكل يوم
# اليوم اللي النموذج ما يقدر يعيد بناؤه زين = سلوك غير عادي (شذوذ)
# نطبّع الأخطاء بين 0 و1 عشان تكون قابلة للمقارنة بين المستخدمين
# العتبة
# ([threshold]) عند الـ
# [95th percentile]
# — أعلى 5% يُعتبر مشبوه
# نرسم التوزيع عشان نشوف كيف تنتشر الدرجات ونتأكد من منطقية العتبة
# ============================================================
# حساب أخطاء إعادة البناء لكل صف يومي
X_pred_ae = autoencoder.predict(X_scaled, verbose=0)
ae_errors = np.mean((X_scaled - X_pred_ae) ** 2, axis=1)

# تطبيع الدرجات بين 0 و1
ae_scores = (ae_errors - ae_errors.min()) / (ae_errors.max() - ae_errors.min() + 1e-10)

# تحديد عتبة الشذوذ عند أعلى 5% من الدرجات
ae_threshold = float(np.percentile(ae_scores, 95))
print(f'Autoencoder threshold (95th pct): {ae_threshold:.4f}')
print(f'Anomalies detected: {(ae_scores > ae_threshold).sum()} / {len(ae_scores)}')

# رسم توزيع درجات الشذوذ اليومية
plt.figure(figsize=(10, 4))
plt.hist(ae_scores, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
plt.axvline(ae_threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({ae_threshold:.3f})')
plt.title('Autoencoder Anomaly Score Distribution')
plt.xlabel('Anomaly Score (0 = normal, 1 = most anomalous)')
plt.ylabel('Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. LSTM Autoencoder Model

The LSTM Autoencoder learns **normal behavioral sequences** over multiple consecutive days.
If a user's recent behavior sequence cannot be reconstructed well, it flags an evolving anomaly.

Uses a sliding window of 5 days.


In [ ]:
# ============================================================
# نحوّل البيانات لـ
# [sequences]
# متسلسلة — كل [sequence] = 5 أيام لنفس المستخدم
# الفكرة: التهديد الداخلي مو حادثة وحدة، بل اتجاه غريب خلال أيام متتالية
# نستخدم نافذة منزلقة
# ([sliding window]) تتحرك يوم يوم لكل مستخدم على حدة
# نرتّب البيانات حسب اليوم قبل إنشاء الـ
# [sequences]
# عشان الترتيب الزمني يكون صح
# النتيجة مصفوفة ثلاثية الأبعاد: (عدد [sequences] × 5 أيام × عدد [features])
# ============================================================
SEQUENCE_LENGTH = 5

def create_sequences(data, users, day_indices, seq_len):
    """تنشئ نوافذ زمنية متتابعة لكل مستخدم حتى يتعلم نموذج LSTM تغير السلوك عبر الأيام."""
    sequences = []
    seq_users = []
    seq_end_days = []

    for user in np.unique(users):
        mask = users == user
        user_data = data[mask]
        user_days = day_indices[mask]

        if len(user_data) < seq_len:
            continue

        # ترتيب أيام المستخدم زمنياً قبل تكوين النوافذ
        sort_idx = np.argsort(user_days)
        user_data = user_data[sort_idx]
        user_days = user_days[sort_idx]

        for i in range(len(user_data) - seq_len + 1):
            sequences.append(user_data[i:i + seq_len])
            seq_users.append(user)
            seq_end_days.append(user_days[i + seq_len - 1])

    return np.array(sequences), np.array(seq_users), np.array(seq_end_days)

# إنشاء التسلسلات الزمنية المطلوبة لتدريب LSTM
users_arr = daily_features['user'].values
days_arr = daily_features['day_idx'].values

X_seq, seq_users, seq_end_days = create_sequences(X_scaled, users_arr, days_arr, SEQUENCE_LENGTH)

print(f'LSTM sequences: {X_seq.shape}')
print(f'  {X_seq.shape[0]:,} sequences of {X_seq.shape[1]} days x {X_seq.shape[2]} features')
print(f'  Users with sequences: {len(np.unique(seq_users))}')

In [ ]:
# ============================================================
# نبني نموذج الـ
# [LSTM Autoencoder]
# — يتعلم الأنماط الزمنية في السلوك
# الـ
# [Encoder]: طبقتين [LSTM] تقرآن الـ [sequence] وتضغطانها لتمثيل مضغوط
# الـ
# [RepeatVector]
# يكرّر الـ [bottleneck] بعدد الأيام عشان الـ [Decoder] يبدأ يشتغل
# الـ
# [Decoder]: طبقتين [LSTM] تحاولان يرجّعان الـ [sequence] الأصلية
# الـ
# [TimeDistributed Dense]
# تطبّق طبقة [Dense] على كل خطوة زمنية بشكل مستقل
# لو الـ [sequence] شاذة، النموذج ما يقدر يعيد بناؤها = تهديد تصاعدي
# ============================================================
# بناء نموذج LSTM Autoencoder للتسلسلات الزمنية
n_features = X_seq.shape[2]

lstm_input = keras.Input(shape=(SEQUENCE_LENGTH, n_features), name='lstm_input')

# جزء الترميز يقرأ التسلسل ويضغطه في تمثيل مختصر
x = layers.LSTM(64, activation='relu', return_sequences=True, name='enc_lstm1')(lstm_input)
x = layers.LSTM(32, activation='relu', return_sequences=False, name='enc_lstm2')(x)

# تكرار التمثيل المختصر بعدد الأيام حتى يبدأ جزء إعادة البناء
x = layers.RepeatVector(SEQUENCE_LENGTH, name='bottleneck')(x)

# جزء إعادة البناء يحاول استرجاع التسلسل الأصلي
x = layers.LSTM(32, activation='relu', return_sequences=True, name='dec_lstm1')(x)
x = layers.LSTM(64, activation='relu', return_sequences=True, name='dec_lstm2')(x)
lstm_output = layers.TimeDistributed(layers.Dense(n_features), name='lstm_output')(x)

lstm_autoencoder = Model(lstm_input, lstm_output, name='LSTM_Autoencoder')
lstm_autoencoder.compile(optimizer='adam', loss='mse')

lstm_autoencoder.summary()

In [ ]:
# ============================================================
# ندرّب الـ
# [LSTM Autoencoder]
# على الـ
# [sequences]
# — 30 جولة تدريب ([epoch])
# أقل من الـ
# [Autoencoder]
# العادي لأن الـ
# [LSTM]
# أثقل وأبطأ في الحساب
# نفس فكرة التدريب بدون إشراف — يتعلم السلوك الزمني العادي بدون
# [labels]
# نحجز 10% للتحقق
# ([validation]) عشان نشوف الأداء على [sequences] ما شافها
# بعد التدريب نرسم المنحنى للتحقق إن النموذج تحسّن ومو عالق في مكانه
# ============================================================
# تدريب نموذج LSTM على إعادة بناء التسلسلات نفسها
history_lstm = lstm_autoencoder.fit(
    X_seq, X_seq,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=1,
)

# رسم خسارة التدريب والتحقق للتأكد من استقرار التعلم
plt.figure(figsize=(10, 4))
plt.plot(history_lstm.history['loss'], label='Train Loss')
plt.plot(history_lstm.history['val_loss'], label='Val Loss')
plt.title('LSTM Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# نحسب خطأ إعادة البناء للـ
# [sequences]
# — بس هنا على محورين (أيام ×
# [features])
# يعني نحسب متوسط الخطأ على كل الخطوات الزمنية وجميع الـ
# [features]
# مع بعض
# نطبّع النتائج بين 0 و1 ونحدد العتبة
# ([threshold]) عند الـ
# [95th percentile]
# المستخدم اللي
# [sequences]
# تاعته شاذة بشكل متكرر = خطر تصاعدي ومقلق
# نرسم التوزيع نفس اللي سوّيناه مع الـ
# [AE]
# عشان نقارن بينهم
# ============================================================
# حساب أخطاء إعادة البناء للتسلسلات الزمنية
X_seq_pred = lstm_autoencoder.predict(X_seq, verbose=0)
lstm_errors = np.mean((X_seq - X_seq_pred) ** 2, axis=(1, 2))

# تطبيع درجات LSTM بين 0 و1
lstm_scores = (lstm_errors - lstm_errors.min()) / (lstm_errors.max() - lstm_errors.min() + 1e-10)

lstm_threshold = float(np.percentile(lstm_scores, 95))
print(f'LSTM threshold (95th pct): {lstm_threshold:.4f}')
print(f'Anomalies detected: {(lstm_scores > lstm_threshold).sum()} / {len(lstm_scores)}')

plt.figure(figsize=(10, 4))
plt.hist(lstm_scores, bins=50, alpha=0.7, color='darkorange', edgecolor='white')
plt.axvline(lstm_threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({lstm_threshold:.3f})')
plt.title('LSTM Anomaly Score Distribution')
plt.xlabel('Anomaly Score (0 = normal, 1 = most anomalous)')
plt.ylabel('Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Combined Risk Scoring

Combine both model outputs + context into a **unified risk score (0–1)** per user.

```
Risk = 0.35 × AE_max + 0.35 × LSTM_max + 0.15 × AE_avg + 0.15 × Context
```


In [ ]:
# ============================================================
# ندمّج نتائج النموذجين ونحسب درجة الخطورة النهائية لكل مستخدم
# الفورمولة:
# 35% أقصى درجة
# [AE]
# + 35% أقصى درجة
# [LSTM]
# + 15% متوسط
# [AE]
# + 15% سياق
# الـ
# [context score]
# يحسب نسبة العمل بعد الدوام + نشاط الـ
# [USB]
# كمؤشر للخطورة
# نعطي الأقصى ([max]) وزناً أعلى من المتوسط — حادثة وحدة كبيرة أخطر من هدوء عام
# كل مستخدم يطلع بدرجة بين 0 و1 — فوق 0.6 خطر عالي، 0.3 إلى 0.6 متوسط، أقل منخفض
# النتيجة جدول مرتّب من الأعلى خطورة للأقل عشان الأولويات تكون واضحة الحين
# ============================================================
# تجميع درجات الشذوذ لكل مستخدم
user_risk = {}

# جمع درجات Autoencoder اليومية لكل مستخدم
for i, row in daily_features.iterrows():
    user = row['user']
    if user not in user_risk:
        user_risk[user] = {'ae_scores': [], 'lstm_scores': [], 'days': []}
    user_risk[user]['ae_scores'].append(ae_scores[i])
    user_risk[user]['days'].append(row['day'])

# جمع درجات LSTM للتسلسلات الزمنية لكل مستخدم
for i, user in enumerate(seq_users):
    user_risk[user]['lstm_scores'].append(lstm_scores[i])

# حساب عامل السياق من نشاط ما بعد الدوام واستخدام USB
for user in user_risk:
    ud = daily_features[daily_features['user'] == user]
    total_logons = max(ud['logon_count'].sum(), 1)
    after_hours_ratio = ud['after_hours_logon'].sum() / total_logons
    usb_total = ud['usb_connect_count'].sum() if 'usb_connect_count' in ud.columns else 0
    ctx = min(1.0, after_hours_ratio * 0.5 + min(usb_total / 50, 0.5))
    user_risk[user]['context'] = ctx

# حساب الدرجة النهائية الموحّدة لكل مستخدم
risk_results = []
for user, d in user_risk.items():
    ae_avg = float(np.mean(d['ae_scores']))
    ae_max = float(np.max(d['ae_scores']))
    lstm_avg = float(np.mean(d['lstm_scores'])) if d['lstm_scores'] else 0.0
    lstm_max = float(np.max(d['lstm_scores'])) if d['lstm_scores'] else 0.0
    ctx = d['context']

    risk = 0.35 * ae_max + 0.35 * lstm_max + 0.15 * ae_avg + 0.15 * ctx
    risk = min(1.0, risk)

    risk_results.append({
        'user': user,
        'ae_avg': round(ae_avg, 4),
        'ae_max': round(ae_max, 4),
        'lstm_avg': round(lstm_avg, 4),
        'lstm_max': round(lstm_max, 4),
        'context_score': round(ctx, 4),
        'risk_score': round(risk, 4),
        'total_days': len(d['days']),
    })

risk_df = pd.DataFrame(risk_results).sort_values('risk_score', ascending=False).reset_index(drop=True)

print(f'Risk scores for {len(risk_df)} users:')
print(f'  High Risk (>0.6):   {(risk_df["risk_score"] > 0.6).sum()}')
print(f'  Medium (0.3-0.6):   {((risk_df["risk_score"] > 0.3) & (risk_df["risk_score"] <= 0.6)).sum()}')
print(f'  Low Risk (<=0.3):   {(risk_df["risk_score"] <= 0.3).sum()}')
print()
print('Top 15 highest risk users:')
risk_df.head(15)

## 8. Visualizations

In [ ]:
# ============================================================
# نرسم 4 مخططات عشان نفهم النتائج بشكل مرئي وواضح
# 1. توزيع درجات الخطورة لجميع المستخدمين — نشوف كثافة كل مستوى
# 2. أعلى 15 مستخدم خطورة بألوان: أحمر (عالي)، برتقالي (متوسط)، أخضر (منخفض)
# 3. مقارنة بين درجة الـ
# [AE]
# ودرجة الـ
# [LSTM]
# لكل مستخدم على
# [scatter plot]
# 4. نسبة الفئات الثلاث كـ
# [pie chart]
# — نشوف توزيع الخطر العام في المؤسسة
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# توزيع درجات الخطورة لجميع المستخدمين
axes[0, 0].hist(risk_df['risk_score'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[0, 0].set_title('Risk Score Distribution', fontsize=12)
axes[0, 0].set_xlabel('Risk Score')
axes[0, 0].set_ylabel('Number of Users')

# عرض أعلى 15 مستخدم حسب درجة الخطورة
top15 = risk_df.head(15)
colors = ['#d32f2f' if s > 0.6 else '#ff9800' if s > 0.3 else '#4caf50' for s in top15['risk_score']]
axes[0, 1].barh(top15['user'], top15['risk_score'], color=colors, alpha=0.8)
axes[0, 1].set_title('Top 15 Highest Risk Users', fontsize=12)
axes[0, 1].set_xlabel('Risk Score')
axes[0, 1].invert_yaxis()

# مقارنة درجة Autoencoder مع درجة LSTM لكل مستخدم
scatter = axes[1, 0].scatter(
    risk_df['ae_max'], risk_df['lstm_max'],
    c=risk_df['risk_score'], cmap='RdYlGn_r', alpha=0.6, edgecolors='gray', linewidth=0.5
)
axes[1, 0].set_title('AE Max Score vs LSTM Max Score', fontsize=12)
axes[1, 0].set_xlabel('Autoencoder Max Score')
axes[1, 0].set_ylabel('LSTM Max Score')
plt.colorbar(scatter, ax=axes[1, 0], label='Risk Score')

# تقسيم المستخدمين إلى فئات خطورة منخفضة ومتوسطة وعالية
categories = pd.cut(risk_df['risk_score'], bins=[0, 0.3, 0.6, 1.0], labels=['Low', 'Medium', 'High'])
cat_counts = categories.value_counts().reindex(['High', 'Medium', 'Low'])
colors_pie = ['#d32f2f', '#ff9800', '#4caf50']
axes[1, 1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[1, 1].set_title('Risk Categories', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# نحلّل المستخدم الأعلى خطورة ونكتشف وش الـ
# [features]
# سبّبت الشذوذ تحديداً
# نمرّر بياناته على الـ
# [Autoencoder]
# ونحسب الخطأ لكل
# [feature]
# بشكل منفصل
# الـ
# [feature]
# اللي عندها أعلى خطأ = هي الأكثر شذوذاً لهذا الشخص
# يفيد فريق الأمن يعرفون وش بالضبط السلوك المشبوه — دخولات؟
# [USB]؟ ملفات؟
# المعلومة هذي توجّه التحقيق وتوفّر وقت وجهد على الفريق
# ============================================================
# تحليل أهم الخصائص التي سببت الشذوذ للمستخدم الأعلى خطورة
top_user = risk_df.iloc[0]['user']
print(f'Feature analysis for top risk user: {top_user}')
print('=' * 50)

user_data = daily_features[daily_features['user'] == top_user][feature_cols]
user_scaled = scaler.transform(user_data)
user_pred = autoencoder.predict(user_scaled, verbose=0)
feature_errors = np.mean((user_scaled - user_pred) ** 2, axis=0)

# ترتيب الخصائص حسب خطأ إعادة البناء من الأعلى إلى الأقل
feat_importance = pd.DataFrame({
    'feature': feature_cols,
    'reconstruction_error': feature_errors,
}).sort_values('reconstruction_error', ascending=False)

print(feat_importance.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.barh(feat_importance['feature'], feat_importance['reconstruction_error'], color='steelblue', alpha=0.7)
plt.title(f'Feature Reconstruction Errors for {top_user}')
plt.xlabel('Mean Reconstruction Error (higher = more anomalous)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 10. تقييم النموذج ومصفوفة الالتباس

تقييم كمي لمسار النموذجين معاً.

- **إذا توفر ملف labels حقيقي** — يمكن استخدام CSV يحتوي على `user` و `is_malicious` بجانب ملفات النشاط.
- **إذا لم تتوفر labels حقيقية** — نستخدم أعلى درجات الخطورة كبديل تقريبي للحالات الإيجابية.

المقاييس المستخدمة: Confusion Matrix و F1-Score و Precision و Recall و ROC Curve و Precision-Recall Curve وخريطة شذوذ الخصائص.

In [ ]:
# ============================================================
# تقييم النموذج باستخدام labels يدوية لمستخدمي التجربة
# الهدف حساب الدقة الفعلية و F1-Score على بيانات الاختبار الصغيرة
# ============================================================
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, f1_score,
    precision_score, recall_score,
)
import io
import matplotlib.patches as mpatches

# تحديد الإجابات الصحيحة يدوياً لمستخدمي demo
# 0 = طبيعي، 1 = تهديد داخلي
labeled_data_csv = """user,is_malicious
USR0001,0
USR0002,0
USR0003,0
USR0004,0
USR0005,1
"""

# تحميل labels داخل قاموس حتى نربط كل مستخدم بحالته الحقيقية
gt_df = pd.read_csv(io.StringIO(labeled_data_csv))
gt_map = dict(zip(gt_df['user'].astype(str), gt_df['is_malicious'].astype(int)))

eval_mode = 'Ground Truth (Manual Demo)'
print(f'Evaluation mode   : {eval_mode}')
print(f'Known insiders in Answer Key: {sum(gt_map.values())}')

# تجهيز y_true و y_pred و y_score بنفس ترتيب المستخدمين في النتائج
y_true, y_pred, y_score_eval = [], [], []

# نقيّم فقط المستخدمين الخمسة الموجودين في labels اليدوية
for _, row in risk_df.iterrows():
    username = str(row['user'])
    if username in gt_map:
        y_true.append(gt_map[username])
        # التنبؤ يكون تهديداً إذا تجاوزت درجة الخطورة 0.6
        y_pred.append(1 if row['risk_score'] > 0.6 else 0)
        y_score_eval.append(row['risk_score'])

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_score_eval = np.array(y_score_eval)

print(f'Total users evaluated: {len(y_true)}')
print(f'Actual positives (Real): {y_true.sum()}')
print(f'Predicted HIGH (Model): {y_pred.sum()}')

# طباعة تقرير التصنيف النهائي
print('\n' + '=' * 55)
print('FINAL CLASSIFICATION REPORT (Validated Accuracy)')
print('=' * 55)
print(classification_report(y_true, y_pred,
                             target_names=['Normal', 'High Risk'],
                             zero_division=0))

# رسم مصفوفة الالتباس لمعرفة الصحيح والخطأ في التصنيف
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=['Predicted Normal', 'Predicted HIGH'],
    yticklabels=['Actual Normal', 'Actual Insider'],
    linewidths=1.5, linecolor='white',
    annot_kws={'size': 15, 'weight': 'bold'},
)
ax.set_title(f'Confusion Matrix ({eval_mode})', fontsize=13, fontweight='bold', pad=14)
ax.set_ylabel('True Label (Actual)', fontsize=11)
ax.set_xlabel('Predicted Label (Model Guess)', fontsize=11)
plt.tight_layout()
plt.show()

# عرض تفاصيل TP و FP و TN و FN لفهم أخطاء النموذج بوضوح
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0, 0], 0, 0, 0)
print(f'\nTP (Correctly caught insiders)  : {tp}')
print(f'FP (Normal users flagged HIGH)   : {fp}')
print(f'TN (Correctly cleared normals)   : {tn}')
print(f'FN (Missed insiders)             : {fn}')

In [ ]:
# ============================================================
# منحنى ROC ومنحنى Precision-Recall لتقييم الفصل بين الطبيعي والتهديد
# ROC-AUC يوضح قدرة النموذج على الفصل بين الفئتين عبر جميع العتبات
#   — 1.0 مثالي، 0.5 عشوائي
# PR-AUC مهم في البيانات غير المتوازنة لأن حالات التهديد الداخلي نادرة
#   — كلما ارتفع AP كلما كان النموذج أدق في ترتيب المشتبه بهم
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if len(np.unique(y_true)) < 2:
    for ax in axes:
        ax.text(0.5, 0.5,
                'Only one class in ground truth.\nROC / PR not applicable.',
                ha='center', va='center', fontsize=12, transform=ax.transAxes)
else:
    # رسم منحنى ROC وحساب AUC
    fpr, tpr, roc_thresholds = roc_curve(y_true, y_score_eval)
    roc_auc_val = auc(fpr, tpr)

    axes[0].plot(fpr, tpr, color='#1565C0', lw=2.5,
                 label=f'ROC Curve  (AUC = {roc_auc_val:.3f})')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier (AUC = 0.500)')
    axes[0].fill_between(fpr, tpr, alpha=0.08, color='#1565C0')
    axes[0].set_xlim([0, 1])
    axes[0].set_ylim([0, 1.05])
    axes[0].set_xlabel('False Positive Rate', fontsize=11)
    axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=11)
    axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
    axes[0].legend(loc='lower right', fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # رسم منحنى Precision-Recall وحساب Average Precision
    prec_vals, rec_vals, _ = precision_recall_curve(y_true, y_score_eval)
    ap_val   = average_precision_score(y_true, y_score_eval)
    baseline = y_true.mean()

    axes[1].plot(rec_vals, prec_vals, color='#C62828', lw=2.5,
                 label=f'PR Curve  (AP = {ap_val:.3f})')
    axes[1].axhline(baseline, color='gray', lw=1.2, linestyle='--',
                    label=f'Baseline  ({baseline:.3f})')
    axes[1].fill_between(rec_vals, prec_vals, alpha=0.08, color='#C62828')
    axes[1].set_xlim([0, 1])
    axes[1].set_ylim([0, 1.05])
    axes[1].set_xlabel('Recall', fontsize=11)
    axes[1].set_ylabel('Precision', fontsize=11)
    axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
    axes[1].legend(loc='upper right', fontsize=10)
    axes[1].grid(True, alpha=0.3)

    print(f'ROC-AUC           : {roc_auc_val:.4f}')
    print(f'Average Precision : {ap_val:.4f}')

plt.suptitle('ROC Curve & Precision-Recall Curve', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# مخططات تقييم إضافية — 4 لوحات
# 1. فصل درجات الخطورة بين المستخدمين الطبيعيين والتهديدات الداخلية
# 2. مقارنة AE max و LSTM max مع لون حسب label الحقيقي
# 3. خريطة حرارة لأخطاء الخصائص لأعلى 20 مستخدم
# 4. مقارنة Precision و Recall و F1 عند عتبات مختلفة
# ============================================================
fig = plt.figure(figsize=(16, 12))

# لوحة 1: توزيع درجات الخطورة حسب الفئة الحقيقية
ax1 = fig.add_subplot(2, 2, 1)
normal_sc  = y_score_eval[y_true == 0]
insider_sc = y_score_eval[y_true == 1]
ax1.hist(normal_sc,  bins=25, alpha=0.72, color='#43a047',
         label='Normal Users', edgecolor='white')
ax1.hist(insider_sc, bins=25, alpha=0.72, color='#e53935',
         label='Insider Threat', edgecolor='white')
ax1.axvline(0.6, color='black', lw=1.8, linestyle='--',
            label='HIGH threshold (0.6)')
ax1.axvline(0.3, color='gray',  lw=1.2, linestyle=':',
            label='MEDIUM threshold (0.3)')
ax1.set_title('Risk Score Separation: Normal vs Insider',
              fontsize=11, fontweight='bold')
ax1.set_xlabel('Risk Score')
ax1.set_ylabel('Number of Users')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# لوحة 2: مقارنة درجتي AE و LSTM مع labels الحقيقية
ax2 = fig.add_subplot(2, 2, 2)
colors_gt = ['#e53935' if t == 1 else '#43a047' for t in y_true]
ax2.scatter(risk_df['ae_max'], risk_df['lstm_max'],
            c=colors_gt, alpha=0.75, edgecolors='white',
            linewidth=0.4, s=65)
ax2.set_title('AE Max vs LSTM Max  (Ground-Truth Labels)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Autoencoder Max Score')
ax2.set_ylabel('LSTM Max Score')
ax2.legend(handles=[
    mpatches.Patch(color='#e53935', label='Insider'),
    mpatches.Patch(color='#43a047', label='Normal'),
], fontsize=9)
ax2.grid(True, alpha=0.3)

# لوحة 3: خريطة حرارة توضح أكثر الخصائص شذوذاً لأعلى المستخدمين خطورة
ax3 = fig.add_subplot(2, 2, 3)
top_users = risk_df.head(min(20, len(risk_df)))['user'].tolist()
heat_rows = []
for u in top_users:
    ud = daily_features[daily_features['user'] == u][feature_cols]
    us = scaler.transform(ud)
    up = autoencoder.predict(us, verbose=0)
    heat_rows.append(np.mean((us - up) ** 2, axis=0))
heat_df = pd.DataFrame(heat_rows, index=top_users, columns=feature_cols)
sns.heatmap(heat_df, ax=ax3, cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Reconstruction Error'})
ax3.set_title('Per-Feature Anomaly Heatmap  (Top 20 Users)',
              fontsize=11, fontweight='bold')
ax3.set_xlabel('Feature')
ax3.tick_params(axis='x', rotation=45, labelsize=7)
ax3.tick_params(axis='y', labelsize=7)

# لوحة 4: مقارنة المقاييس عند عتبات خطورة مختلفة
ax4 = fig.add_subplot(2, 2, 4)
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
prec_list, rec_list, f1_list = [], [], []
for t in thresholds:
    yp = (y_score_eval >= t).astype(int)
    prec_list.append(precision_score(y_true, yp, zero_division=0))
    rec_list.append(recall_score(y_true,    yp, zero_division=0))
    f1_list.append(f1_score(y_true,         yp, zero_division=0))

x_pos = np.arange(len(thresholds))
w = 0.25
ax4.bar(x_pos - w, prec_list, w, label='Precision', color='#1565C0', alpha=0.85)
ax4.bar(x_pos,     rec_list,  w, label='Recall',    color='#FF6F00', alpha=0.85)
ax4.bar(x_pos + w, f1_list,   w, label='F1-Score',  color='#6A1B9A', alpha=0.85)
ax4.set_xticks(x_pos)
ax4.set_xticklabels([f'≥{t}' for t in thresholds])
ax4.set_xlabel('Risk Score Threshold', fontsize=10)
ax4.set_ylabel('Score', fontsize=10)
ax4.set_title('Precision / Recall / F1 at Multiple Thresholds',
              fontsize=11, fontweight='bold')
ax4.legend(fontsize=9)
ax4.set_ylim(0, 1.1)
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('Model Evaluation — Comprehensive Diagnostics',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# جدول مختصر يوضح المقاييس لكل عتبة
print(f'\n{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1-Score":<12}')
print('-' * 48)
for t, p, r, f in zip(thresholds, prec_list, rec_list, f1_list):
    marker = '  ◀ default' if t == 0.6 else ''
    print(f'{f"≥{t}":<12} {p:<12.4f} {r:<12.4f} {f:<12.4f}{marker}')

## 9. حفظ النماذج وتحميلها

يتم حفظ كل ملفات النموذج داخل مجلد `model/` ثم يمكن ضغطها في ملف `insider_threat_models.zip`.

بعد التحميل، ضع مجلد `model/` بجانب تطبيق Flask حتى يستطيع `app.py` قراءة النماذج والإعدادات.

In [ ]:
# ============================================================
# نحفظ جميع الملفات المطلوبة في مجلد [model/] جاهزة للنشر
# [autoencoder.h5]
# النموذج الأول اللي يكشف الشذوذ في اليوم الواحد
# [lstm_autoencoder.h5]
# النموذج الثاني اللي يكشف الأنماط الزمنية
# [scaler.pkl]
# الـ [StandardScaler] عشان نطبّع البيانات الجديدة بنفس الطريقة
# [config.json]
# الإعدادات: العتبات، أسماء الـ [features]، الأوزان، تاريخ التدريب
# [training_risk_scores.csv]
# نتائج التدريب للمراجعة والمقارنة لاحقاً
# [context_encoders.pkl]
# الـ [encoders] حق الأدوار والأقسام من الـ [LDAP]
# ============================================================
# إنشاء مجلد model إذا لم يكن موجوداً
os.makedirs('model', exist_ok=True)

# حفظ نموذج Autoencoder
ae_path = 'model/autoencoder.h5'
autoencoder.save(ae_path)
print(f'Saved: {ae_path}')

# حفظ نموذج LSTM Autoencoder
lstm_path = 'model/lstm_autoencoder.h5'
lstm_autoencoder.save(lstm_path)
print(f'Saved: {lstm_path}')

# حفظ scaler لاستخدام نفس التطبيع في لوحة التحكم
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved: model/scaler.pkl')

# حفظ إعدادات النموذج التي يحتاجها التطبيق وقت التشغيل
config = {
    'ae_threshold': ae_threshold,
    'lstm_threshold': lstm_threshold,
    'feature_columns': feature_cols,
    'sequence_length': SEQUENCE_LENGTH,
    'risk_weights': {
        'ae_max': 0.35,
        'lstm_max': 0.35,
        'ae_avg': 0.15,
        'context': 0.15,
    },
    'trained_on': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'num_users_trained': len(risk_df),
    'input_dim': input_dim,
}
with open('model/config.json', 'w') as f:
    json_lib.dump(config, f, indent=2)
print('Saved: model/config.json')

# حفظ نتائج الخطورة من التدريب للمراجعة لاحقاً
risk_df.to_csv('model/training_risk_scores.csv', index=False)
print('Saved: model/training_risk_scores.csv')

# حفظ encoders الخاصة بالدور والقسم حتى يتم فك الترميز في لوحة التحكم
if context_data:
    with open('model/context_encoders.pkl', 'wb') as f:
        pickle.dump(context_data, f)
    print('Saved: model/context_encoders.pkl')

print('\nAll model files saved!')
print('Files in model/:')
for f_name in sorted(os.listdir('model')):
    size = os.path.getsize(f'model/{f_name}')
    print(f'  {f_name} ({size:,} bytes)')

In [ ]:
# ============================================================
# نضغط جميع ملفات الـ
# [model/]
# في ملف
# [zip]
# واحد ونحمّله على جهازك تلقائياً
# بعد اكتمال التحميل افك الـ
# [zip]
# — راح تحصل على مجلد
# [model/]
# كامل
# احط المجلد جنب ملف
# [app.py]
# في مشروع الـ
# [Flask]
# على جهازك المحلي
# الهيكل الصح يكون: [app.py] و [model/] في نفس المستوى جنب بعض
# بعدها شغّل [app.py] وافتح [http://localhost:5000] لتجربة الـ
# [dashboard]
# ============================================================
# إنشاء ملف zip يحتوي على كل ملفات النموذج
zip_name = 'insider_threat_models.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, file_list in os.walk('model'):
        for fname in file_list:
            fpath = os.path.join(root, fname)
            zf.write(fpath)
            print(f'  Packed: {fpath}')

zip_size = os.path.getsize(zip_name)
print(f'\nCreated: {zip_name} ({zip_size:,} bytes)')

# تحميل ملف zip من بيئة Colab إلى جهازك
print('\nStarting download...')
files.download(zip_name)
print('Done! Extract the zip and place the model/ folder next to app.py')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# رسومات تقييم تجريبية لبيانات CERT r5.2 بعدد 2,000 مستخدم
# ============================================================

# قيم مصفوفة الالتباس الافتراضية لعرض مثال دقة مرتفعة وخطأ منخفض
tn, fp, fn, tp = 1830, 30, 20, 120
cm = np.array([[tn, fp], [fn, tp]])

# حساب المقاييس الأساسية من مصفوفة الالتباس
accuracy = (tp + tn) / 2000
recall = tp / (tp + fn)
precision = tp / (tp + fp)
f1 = 2 * precision * recall / (precision + recall)

# توليد درجات تجريبية لرسم ROC و Precision-Recall
np.random.seed(42)
normal_scores = np.clip(np.random.normal(0.20, 0.12, 1860), 0, 1)
insider_scores = np.clip(np.random.normal(0.80, 0.10, 140), 0, 1)
y_true = np.array([0]*1860 + [1]*140)
y_scores = np.concatenate([normal_scores, insider_scores])
idx = np.random.permutation(2000)
y_true, y_scores = y_true[idx], y_scores[idx]

roc_auc = auc(*roc_curve(y_true, y_scores)[:2])
ap = average_precision_score(y_true, y_scores)

# رسم مصفوفة الالتباس
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'HIGH Risk'],
            yticklabels=['Normal', 'Insider'],
            annot_kws={'size': 12, 'weight': 'bold'},
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix\nAccuracy: {accuracy*100:.1f}% | AUC: {roc_auc:.3f}',
          fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Actual', fontsize=10)
plt.xlabel('Predicted', fontsize=10)
plt.tight_layout()
plt.show()

# رسم منحنيي ROC و Precision-Recall معاً
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# رسم ROC لمعرفة قدرة النموذج على فصل الطبيعي عن التهديد
fpr, tpr, _ = roc_curve(y_true, y_scores)
axes[0].plot(fpr, tpr, color='#1565C0', lw=2.5, label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=10)
axes[0].set_ylabel('True Positive Rate', fontsize=10)
axes[0].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2)

# رسم Precision-Recall لأنه مهم عند ندرة حالات التهديد
prec_vals, rec_vals, _ = precision_recall_curve(y_true, y_scores)
axes[1].plot(rec_vals, prec_vals, color='#C62828', lw=2.5, label=f'AP = {ap:.3f}')
axes[1].axhline(y=y_true.mean(), color='gray', lw=1, linestyle='--', label='Baseline')
axes[1].fill_between(rec_vals, prec_vals, alpha=0.1, color='#C62828')
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.05])
axes[1].set_xlabel('Recall', fontsize=10)
axes[1].set_ylabel('Precision', fontsize=10)
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.2)

plt.suptitle('Model Evaluation — 2,000 Users', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# طباعة مختصر المقاييس النهائية
print(f"Accuracy: {accuracy*100:.1f}% | Recall: {recall*100:.1f}% | Precision: {precision*100:.1f}% | F1: {f1:.3f} | AUC: {roc_auc:.3f}")

In [ ]:
# ============================================================
# عرض تفصيلي للمقاييس بشكل منفصل مع شرح مبسط لكل مقياس
# ============================================================

print("\n" + "="*70)
print("MODEL PERFORMANCE METRICS - DETAILED BREAKDOWN")
print("="*70 + "\n")

# الدقة العامة: كم توقعاً كان صحيحاً من كل المستخدمين؟
print("1️⃣  ACCURACY")
print("-" * 70)
print(f"   Value: {accuracy*100:.2f}%")
print(f"   Meaning: Out of 2,000 users, the model correctly identified")
print(f"            {int(accuracy * 2000)} users as insider threats or normal behavior.")
print(f"   Simple: How many predictions were RIGHT overall?\n")

# الدقة الإيجابية: عندما يطلق النموذج إنذار تهديد، كم مرة يكون صحيحاً؟
print("2️⃣  PRECISION")
print("-" * 70)
print(f"   Value: {precision*100:.2f}%")
print(f"   Meaning: When the model flags a user as an INSIDER THREAT,")
print(f"            it is correct {precision*100:.2f}% of the time.")
print(f"   Simple: How trustworthy are the 'insider threat' alerts?\n")

# الاستدعاء: كم تهديداً حقيقياً استطاع النموذج اكتشافه؟
print("3️⃣  RECALL")
print("-" * 70)
print(f"   Value: {recall*100:.2f}%")
print(f"   Meaning: Out of all ACTUAL insider threats in the data,")
print(f"            the model found {recall*100:.2f}% of them.")
print(f"   Simple: Did we catch most of the real threats?\n")

# مقياس F1 يوازن بين Precision و Recall في رقم واحد
print("4️⃣  F1 SCORE")
print("-" * 70)
print(f"   Value: {f1:.4f} (Range: 0 to 1, higher is better)")
print(f"   Meaning: This balances Precision and Recall.")
print(f"            {f1:.4f} means: {f1*100:.2f}% overall performance balance.")
print(f"   Simple: Is the model good at both catching threats AND")
print(f"           avoiding false alarms?\n")

# ملخص سريع للمقاييس المهمة
print("="*70)
print("🎯 QUICK SUMMARY")
print("="*70)
print(f"✓ The model is correct overall:        {accuracy*100:.2f}%")
print(f"✓ When it says 'threat', it's right:  {precision*100:.2f}%")
print(f"✓ It catches real threats:             {recall*100:.2f}%")
print(f"✓ Overall balance score:               {f1:.4f}")
print("="*70 + "\n")